# Fine-Tuning DistilBERT for Sentiment Analysis on IMDb Reviews


The goal of this project is to fine-tune a pre-trained Large Language Model (LLM) to classify movie reviews as **positive** or **negative**.

In this project, a small, pre-trained language model called DistilBERT is taught to read movie reviews and decide whether each one is positive or negative. I use the IMDb reviews dataset and the Hugging Face library to fine-tune the model. Then it should be checked how well it works using accuracy and F1 score and use it on simple sentences.

Training was done on a GPU (NVIDIA T4) because fine-tuning a transformer on a CPU would take many hours.

## Step 1: Setup and load the IMDb dataset


1. Install useful libraries

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

2. Checking the versions

In [ ]:
import transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

transformers: 5.20.2
datasets: 4.8.5


3. Load the IMDb Dataset

   I used the `IMDb Movie Review dataset` from Hugging Face. It contains:
   - 25,000 training reviews
   - 25,000 test reviews

In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

# One training example
example = dataset["train"][0]
print("Label:", example["label"])
print("Text:", example["text"][:300])

from collections import Counter
print(Counter(dataset["train"]["label"]))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Label: 0
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h
Counter({0: 12500, 1: 12500})


## Step 2: Tokenization

 Neural networks cannot read text directly, so the text must be converted into numbers. This is done by a **Tokenization**. It converts words into numerical IDs so a model can process, analyze, and run math on the text.

1. Load the tokenizer and trying it on an example sentence

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

sample = "This movie was absolutely wonderful!"
encoded = tokenizer(sample)

print(encoded)
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 2023, 3185, 2001, 7078, 6919, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', 'this', 'movie', 'was', 'absolutely', 'wonderful', '!', '[SEP]']


2. Checking the length of reviews

   **Truncation:** When I checked the review lengths, the average was about 315 tokens, and about 45% of the reviews were longer than 256 tokens. DistilBERT can handle at most 512 tokens. I truncated reviews to **256 tokens** to make training faster and use less memory.

In [ ]:
lengths = [len(tokenizer(t)["input_ids"]) for t in dataset["train"]["text"][:1000]]
print("Average length:", sum(lengths) / len(lengths))
print("Longest:", max(lengths))
print("Reviews over 256 tokens:", sum(l > 256 for l in lengths), "out of 1000")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


Average length: 315.091
Longest: 1525
Reviews over 256 tokens: 455 out of 1000


3. Tokenizing the whole dataset

In [ ]:
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tokenized = dataset.map(tokenize_function, batched=True)
print(tokenized)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


4. Checking the result

In [ ]:
print(tokenized["train"][0].keys())
print(tokenized["train"][0]["input_ids"][:20])
print(len(tokenized["train"][0]["input_ids"]))

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])
[101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009]
256


## Step 3: Load the model and run a test training

1. Load the model
   
   This takes the pre-trained DistilBERT and adds a new, randomly initialised **classification layer** on top, which outputs a score for "negative" and a score for "positive".

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2. Set up the metrics

In [ ]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"],
    }

3. Make small subsets for the test run

In [ ]:
small_train = tokenized["train"].shuffle(seed=42).select(range(1000))
small_test = tokenized["test"].shuffle(seed=42).select(range(500))
print(len(small_train), len(small_test))

1000 500


4. Test training on the small subset

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

test_args = TrainingArguments(
    output_dir="test_run",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    report_to="none",
)

test_trainer = Trainer(
    model=model,
    args=test_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

test_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.615611,0.724000,0.672986


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=0.660057855030847, metrics={'train_runtime': 36.4749, 'train_samples_per_second': 27.416, 'train_steps_per_second': 1.727, 'total_flos': 66233699328000.0, 'train_loss': 0.660057855030847, 'epoch': 1.0})

## Step 4: Full fine-tuning

Summary:

1. The model starts with its pre-trained weights, which already understand English.
2. It is trained on labelled movie reviews with a small learning rate (2e-5), which adjusts the weights slightly toward the sentiment task rather than overwriting what it already knows.
3. The new classification layer learns to map the model's understanding of a review to "positive" or "negative".

1. Reloading a fresh model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2. Set up the full training

In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert-imdb",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

3. Train

    I used the Hugging Face `Trainer` API, which handles the training loop, batching, evaluation and saving.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.242410,0.258923,0.897760,0.892043
2,0.154563,0.269514,0.913200,0.914073


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.22946743711934056, metrics={'train_runtime': 440.2315, 'train_samples_per_second': 113.577, 'train_steps_per_second': 7.101, 'total_flos': 3311668407975168.0, 'train_loss': 0.22946743711934056, 'epoch': 2.0})

4. Save the model

In [ ]:
trainer.save_model("final_model")
tokenizer.save_pretrained("final_model")
print("Saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r final_model /content/drive/MyDrive/final_model

Mounted at /content/drive


## Step 5: Evaluation and prediction

1. Final evaluation on the test set

   The model was evaluated on the full test set of 25,000 reviews that it did not train on.

In [ ]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.154563,0.269514,2,0.913200,0.914073


{'eval_loss': 0.2695136070251465, 'eval_accuracy': 0.9132, 'eval_f1': 0.9140730181357408}


2. Detailed breakdown

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(tokenized["test"])
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

print(classification_report(y_true, y_pred, target_names=["NEGATIVE", "POSITIVE"]))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

    NEGATIVE       0.92      0.90      0.91     12500
    POSITIVE       0.90      0.92      0.91     12500

    accuracy                           0.91     25000
   macro avg       0.91      0.91      0.91     25000
weighted avg       0.91      0.91      0.91     25000

Confusion matrix:
[[11288  1212]
 [  958 11542]]


3. The prediction function

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="final_model",
    tokenizer="final_model",
    device=0,
    truncation=True,
    max_length=256,
)

def predict_sentiment(text):
    result = classifier(text)[0]
    return f"{result['label']} (confidence: {result['score']:.2%})"

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

4. Trying it on new sentences

In [ ]:
tests = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "Terrible film. The plot made no sense and the acting was awful.",
    "It was okay, nothing special but not bad either.",
    "Oh great, another predictable sequel. Just what the world needed.",
    "I didn't hate it, but I wouldn't watch it again.",
]

for t in tests:
    print(t)
    print("  ->", predict_sentiment(t))
    print()

This movie was absolutely fantastic, I loved every minute of it!
  -> POSITIVE (confidence: 99.64%)

Terrible film. The plot made no sense and the acting was awful.
  -> NEGATIVE (confidence: 99.62%)

It was okay, nothing special but not bad either.
  -> NEGATIVE (confidence: 61.55%)

Oh great, another predictable sequel. Just what the world needed.
  -> NEGATIVE (confidence: 95.96%)

I didn't hate it, but I wouldn't watch it again.
  -> NEGATIVE (confidence: 87.67%)



5. Testing on challenging sentences

   To further evaluate the model's robustness, I tested it on more challenging sentences containing mixed sentiments, sarcasm, or complex structures where sentiment is nuanced rather than straightforward.

In [ ]:
hard_tests = [
    "The cinematography was gorgeous, but the story bored me to tears.",
    "I expected a masterpiece. Instead I got two hours of my life I'll never get back.",
    "Not bad at all, I actually enjoyed it more than I expected.",
    "A wonderful cast completely wasted on a script written by a toddler.",
]
for t in hard_tests:
    print(t, "\n  ->", predict_sentiment(t), "\n")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


The cinematography was gorgeous, but the story bored me to tears. 
  -> POSITIVE (confidence: 52.77%) 

I expected a masterpiece. Instead I got two hours of my life I'll never get back. 
  -> POSITIVE (confidence: 87.77%) 

Not bad at all, I actually enjoyed it more than I expected. 
  -> POSITIVE (confidence: 99.52%) 

A wonderful cast completely wasted on a script written by a toddler. 
  -> NEGATIVE (confidence: 99.38%) 



## Step 6: Conclusion - Powers and Limitations of Pre-trained LLMs

### Summary of Results
After fine-tuning **DistilBERT** on the IMDb dataset for 2 epochs, the model achieved impressive evaluation metrics on the 25,000 unseen test reviews:
- **Accuracy:** 91.32%
- **F1-Score:** 91.41%

---

### The Power of Pre-trained LLMs (Transfer Learning)
1. **Data Efficiency:** Training a model from scratch requires millions of texts and massive compute. Through transfer learning, DistilBERT already understood English grammar, syntax, and semantic nuances from its pre-training corpus. This allowed it to achieve over 90% accuracy quickly using only a small learning rate ($\text{2e-5}$).
2. **Contextual Understanding:** Unlike classical machine learning models (like Bag-of-Words or TF-IDF paired with Logistic Regression), transformer models capture word order, context, and subtle negations (e.g., understanding how words change meaning when paired together).
3. **Fast Adaptation:** With Hugging Face’s `Trainer` API and GPU acceleration (NVIDIA T4), the entire training process finished smoothly in minutes.

---

### Limitations of Pre-trained LLMs
1. **Nuance and Complex Sentiment:** While simple sentences yielded near 100% confidence, mixed or sarcastic inputs (e.g., *"The cinematography was gorgeous, but the story bored me to tears"* or *"I expected a masterpiece. Instead I got two hours of my life I'll never get back"*) can confuse the binary classification boundary, sometimes resulting in lower confidence scores or misclassifications because they blend positive and negative traits.
2. **Domain Shift:** A model fine-tuned on movie reviews will perform well on cinematic language, but its performance may degrade if deployed to entirely different domains (like medical diagnoses, financial reports, or legal contracts) without domain-specific fine-tuning.
3. **Resource Dependency:** Even though DistilBERT is a "small" LLM designed to be lightweight, running inferences at scale or training larger models still demands specialized hardware (GPUs/TPUs).